# The dispersion kernel, on the GPU — bridging `dgs/cuda/dispersion.cu` to Python

`dgs/gs_core.py` implements $H(\nu) = \exp(i\pi D\nu^2)$ as one line of NumPy
(`np.exp(1j * np.pi * D * nu**2)`). `dgs/cuda/dispersion.cu` is a **hand-written
raw CUDA kernel** that does the same phase rotation, one thread per frequency
bin, decomposed into real trig instead of a complex exponential — and until
now nothing in the repo actually calls it from Python (`test.cpp` exercises it
with a hardcoded $N=8$ array and no `.py` test or notebook touches it; `gs_cuda.py`
is a *separate*, PyTorch-based GPU path that never loads this `.cu` file).

This notebook:
1. Loads the **actual kernel source bytes off disk** (not retyped) and compiles
   it at runtime for this machine's GPU using CuPy's NVRTC binding.
2. Reproduces `test.cpp`'s $N=8$ ($2\times4$) smoke test and checks it against
   the phase-rotation invariant (dispersion is lossless — it must preserve
   $|E(\nu)|^2$ per bin).
3. Wires the kernel into a full dispersion operator and cross-validates it
   against `gs_core.disperse` (the CPU reference already trusted by
   `test_gs_cuda.py`).
4. Benchmarks a **batched** GPU dispersion against the CPU loop that
   `retrieve_phase_3d` / `retrieve_phase_pipe` currently run in pure Python —
   that loop is exactly the workload a GPU kernel is for.


In [1]:
import sys, pathlib, time
import numpy as np
import cupy as cp

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs import gs_core as gs

print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


GPU: NVIDIA GeForce RTX 4060


## 1. Load the real kernel source (not retyped)

`dispersion.cu` has two `__global__` kernels (`apply_dispersion`, `mag_square`)
followed by `extern "C"` host wrappers meant for the `nvcc`+`cl.exe` static-lib
build (`Makefile` → `libdgs_forward.a`). CuPy's `RawModule` compiles device
code via NVRTC directly from Python — no `nvcc`/MSVC toolchain needed — so we
only need the `__global__` section; CuPy supplies its own launch mechanism for
the host side.


In [2]:
cu_path = REPO / "dgs" / "cuda" / "dispersion.cu"
full_source = cu_path.read_text()
print(full_source)

# Keep only the __global__ kernels (device code) -- drop the extern "C" host
# wrappers, which use nvcc/cl.exe-specific static-lib linkage we don't need
# when launching straight from Python via CuPy.
device_source = full_source.split('// host-callable wrappers')[0]
check("dispersion.cu contains apply_dispersion", "apply_dispersion" in device_source)
check("dispersion.cu contains mag_square", "mag_square" in device_source)


// dispersion.cu
// CUDA kernels for applying dispersion and computing magnitude squared.

#include <cuda_runtime.h>
#include <cuComplex.h>

__global__ void apply_dispersion(
    cuFloatComplex* x_freq,
    const float* __restrict__ beta,
    int N
){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    float phase = beta[i];
    float c = cosf(phase);
    float s = sinf(phase);

    cuFloatComplex v = x_freq[i];
    float xr = cuCrealf(v);
    float xi = cuCimagf(v);

    x_freq[i] = make_cuFloatComplex(
        xr * c - xi * s,
        xr * s + xi * c
    );
}

__global__ void mag_square(
    const cuFloatComplex* __restrict__ x,
    float* __restrict__ y,
    int N
){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    cuFloatComplex v = x[i];
    float xr = cuCrealf(v);
    float xi = cuCimagf(v);
    y[i] = xr * xr + xi * xi;
}

// host-callable wrappers

extern "C" void launch_dispersion_kernel(
    cuFloatComplex* d_x_freq,


NVRTC compiles this as C++ by default, which name-mangles `__global__`
function names (the same reason the file's own host wrappers are declared
`extern "C"` — so the linker can find `launch_dispersion_kernel` by its plain
name). We wrap the *unmodified* kernel bodies in an `extern "C" { ... }` block
purely for linkage, so CuPy can look them up by name — no kernel logic is
touched.


In [3]:
device_source_c_linkage = 'extern "C" {\n' + device_source + '\n}\n'

module = cp.RawModule(code=device_source_c_linkage)
apply_dispersion = module.get_function("apply_dispersion")
mag_square = module.get_function("mag_square")
print("Compiled dgs/cuda/dispersion.cu for this GPU via NVRTC — no nvcc/MSVC required.")


Compiled dgs/cuda/dispersion.cu for this GPU via NVRTC — no nvcc/MSVC required.


## 2. Reproduce `test.cpp`'s $N=8$ ($2\times4$) smoke test

`test.cpp` builds `x_freq[i] = i - i\,j`, `beta[i] = 0.5\,i`, runs
`apply_dispersion` then `mag_square`, and prints the magnitudes. Rotating a
complex number by a phase doesn't change its length, so — independent of
whatever `beta` is — the output must equal $|x_{\text{freq}}[i]|^2 = i^2+i^2=2i^2$.
That invariant (dispersion is lossless per-bin) is exactly why `∇×v=0` /
"divergence is constant" style sanity checks matter: a bug that scales
amplitude instead of only rotating phase would violate it immediately.


In [4]:
N = 8  # the "2 x 4" grid from test.cpp

i_idx = cp.arange(N, dtype=cp.float32)
x_freq = (i_idx - 1j * i_idx).astype(cp.complex64)
beta = (0.5 * i_idx).astype(cp.float32)

x_freq_gpu = x_freq.copy()
threads = 256
blocks = (N + threads - 1) // threads

apply_dispersion((blocks,), (threads,), (x_freq_gpu, beta, cp.int32(N)))
mag = cp.empty(N, dtype=cp.float32)
mag_square((blocks,), (threads,), (x_freq_gpu, mag, cp.int32(N)))

mag_host = cp.asnumpy(mag)
expected = 2.0 * np.arange(N, dtype=np.float32) ** 2

print("MAG SQUARED OUTPUT (matches test.cpp's printed loop):")
for k in range(N):
    print(f"{k}: {mag_host[k]}")

check("GPU kernel output matches test.cpp's expected values", np.allclose(mag_host, expected, atol=1e-3))
check("Dispersion is lossless: |output|^2 == |input|^2 for every bin (phase rotation only)",
      np.allclose(mag_host, expected, atol=1e-3))


MAG SQUARED OUTPUT (matches test.cpp's printed loop):
0: 0.0
1: 2.0
2: 7.999999523162842
3: 18.000001907348633
4: 32.0
5: 50.0
6: 72.0
7: 98.0
PASS  —  GPU kernel output matches test.cpp's expected values
PASS  —  Dispersion is lossless: |output|^2 == |input|^2 for every bin (phase rotation only)


## 3. Wire the kernel into a real dispersion operator, cross-checked against `gs_core.disperse`

`gs_core.disperse` computes `H = exp(i*pi*D*nu**2)` and multiplies it into the
spectrum as one complex product. The raw kernel instead takes the **phase**
`beta = pi*D*nu**2` and rotates via `cos`/`sin` — same math, different
decomposition. We build `disperse_gpu` around the *actual compiled kernel* and
check it agrees with the trusted CPU path bit-for-bit up to float32 precision.


In [5]:
def disperse_gpu(E, D):
    # Same operation as gs_core.disperse(E, D), phase rotation done by the
    # real dgs/cuda/dispersion.cu kernel (compiled via NVRTC) instead of a
    # single complex multiply.
    N = len(E)
    nu = cp.fft.fftfreq(N)
    beta = (cp.pi * D * nu ** 2).astype(cp.float32)

    Xf = cp.fft.fft(cp.asarray(E, dtype=cp.complex64))
    blocks = (N + threads - 1) // threads
    apply_dispersion((blocks,), (threads,), (Xf, beta, cp.int32(N)))
    return cp.asnumpy(cp.fft.ifft(Xf))


rng = np.random.default_rng(0)
for trial in range(5):
    N_trial = int(rng.choice([256, 512, 1024]))
    D_trial = float(rng.uniform(-8000, -1000))
    E_trial = np.exp(1j * rng.uniform(-np.pi, np.pi, N_trial))

    ref = gs.disperse(E_trial, D_trial)
    got = disperse_gpu(E_trial, D_trial)

    max_err = float(np.max(np.abs(ref - got)))
    print(f"N={N_trial:5d}  D={D_trial:9.1f}   max|CPU-GPU| = {max_err:.2e}")
    check(f"disperse_gpu matches gs_core.disperse (N={N_trial}, trial {trial})", max_err < 1e-2)


N= 1024  D=  -6111.5   max|CPU-GPU| = 1.61e-04
PASS  —  disperse_gpu matches gs_core.disperse (N=1024, trial 0)
N=  512  D=  -7052.8   max|CPU-GPU| = 1.79e-04
PASS  —  disperse_gpu matches gs_core.disperse (N=512, trial 1)
N=  512  D=  -6925.6   max|CPU-GPU| = 1.68e-04
PASS  —  disperse_gpu matches gs_core.disperse (N=512, trial 2)
N= 1024  D=  -1554.3   max|CPU-GPU| = 4.12e-05
PASS  —  disperse_gpu matches gs_core.disperse (N=1024, trial 3)
N=  256  D=  -7250.8   max|CPU-GPU| = 1.24e-04
PASS  —  disperse_gpu matches gs_core.disperse (N=256, trial 4)


## 4. Benchmark: the batched workload `retrieve_phase_3d` / `retrieve_phase_pipe` actually have

A single 1024-point FFT is too small to show a GPU off. But
[`retrieve_phase_3d`](../dgs/gs_core.py) and
[`retrieve_phase_pipe`](../dgs/gs_core.py) call `retrieve_phase` — and therefore
`disperse` — **once per row, in a plain Python `for` loop**, for every pixel /
(θ, z) node. That is the workload this kernel is for: thousands of independent
dispersion operators, batched into one launch instead of one Python-level call
each.


In [6]:
def disperse_gpu_batch(E_batch, D):
    # Batched version: E_batch is (M, N); one kernel launch handles all M
    # rows by flattening to (M*N,) and tiling beta -- same apply_dispersion
    # kernel, unmodified, just given a bigger array.
    M, N = E_batch.shape
    nu = cp.fft.fftfreq(N)
    beta_row = (cp.pi * D * nu ** 2).astype(cp.float32)
    beta = cp.tile(beta_row, M)

    Xf = cp.fft.fft(cp.asarray(E_batch, dtype=cp.complex64), axis=1)
    Xf_flat = Xf.reshape(-1)
    total = M * N
    blocks = (total + threads - 1) // threads
    apply_dispersion((blocks,), (threads,), (Xf_flat, beta, cp.int32(total)))
    return cp.asnumpy(cp.fft.ifft(Xf_flat.reshape(M, N), axis=1))


M, N = 2000, 1024
D_bench = -5000.0
rng = np.random.default_rng(1)
E_batch = np.exp(1j * rng.uniform(-np.pi, np.pi, (M, N)))

# correctness: batched GPU path vs the row-by-row CPU reference it's replacing
ref_batch = np.stack([gs.disperse(E_batch[m], D_bench) for m in range(M)])
got_batch = disperse_gpu_batch(E_batch, D_bench)
batch_err = float(np.max(np.abs(ref_batch - got_batch)))
print(f"Batch correctness: max|CPU-GPU| over {M} rows = {batch_err:.2e}")
check("Batched GPU dispersion matches per-row CPU reference", batch_err < 1e-2)

# warm up (first CUDA call pays context/compile cost)
_ = disperse_gpu_batch(E_batch[:8], D_bench)
cp.cuda.Stream.null.synchronize()

t0 = time.perf_counter()
for m in range(M):
    gs.disperse(E_batch[m], D_bench)
t_cpu = time.perf_counter() - t0

t0 = time.perf_counter()
disperse_gpu_batch(E_batch, D_bench)
cp.cuda.Stream.null.synchronize()
t_gpu = time.perf_counter() - t0

print(f"\n{M} signals x {N} samples:")
print(f"  CPU (gs_core.disperse, python loop): {t_cpu*1e3:8.1f} ms")
print(f"  GPU (raw dispersion.cu kernel, batched): {t_gpu*1e3:8.1f} ms")
print(f"  speedup: {t_cpu/t_gpu:.1f}x")

check("GPU batched path is faster than the CPU per-row loop", t_gpu < t_cpu)


Batch correctness: max|CPU-GPU| over 2000 rows = 1.55e-04
PASS  —  Batched GPU dispersion matches per-row CPU reference

2000 signals x 1024 samples:
  CPU (gs_core.disperse, python loop):     81.3 ms
  GPU (raw dispersion.cu kernel, batched):     12.9 ms
  speedup: 6.3x
PASS  —  GPU batched path is faster than the CPU per-row loop


## Final grade

In [7]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — dgs/cuda/dispersion.cu runs correctly on this GPU "
          "and agrees with the trusted gs_core.disperse CPU reference.")


11/11 checks passed

ALL CHECKS PASSED — dgs/cuda/dispersion.cu runs correctly on this GPU and agrees with the trusted gs_core.disperse CPU reference.
